# 프롬프트 (`Prompt`)
`02_prompt.ipynb`

- LLM한테 주는 입력(지시, 맥락, 기억)
    - 지시: '~~해줘'
    - 맥락: Context - 현재 지시를 위해 제공하는 추가 정보
    - 기억: Memory - 지금까지 했던 대화 내용

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## `PromptTemplate`

- 단순히 1회성 명령을 내릴 때 사용

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model='gpt-4.1-nano')

In [3]:
# 추후에 체인.invoke에서 {} 내부에 들어갈 말을 채워줘야 함.
template = '{country}의 수도가 어디인가요?'

# .from_template 메서드로 프롬프트 만들기
prompt = PromptTemplate.from_template(template)

# {}를 채우는 방법 (우리는 결국 chain.invoke 로 쓰게 됨)
prompt.format(country='대한민국')

'대한민국의 수도가 어디인가요?'

In [4]:
chain = prompt | llm

chain.invoke({'country': '대한민국'}).content

'대한민국의 수도는 서울입니다.'

### Partial Variable (부분 변수)

- 프롬프트에서 매개변수 기본값 사용하기

In [5]:
template = '{c1}과 {c2}의 수도는 각각 어디인가요?'

prompt = PromptTemplate(
    template=template,
    input_variables=['c1'],
    partial_variables={
        'c2': '미국'
    }
)

prompt.format(c1='한국')
# prompt.format(c2='캐나다') 

'한국과 미국의 수도는 각각 어디인가요?'

In [6]:
chain = prompt | llm
chain.invoke({'c1': 'kor', 'c2': 'us'}).content

'한국(코리아)의 수도는 서울이고, 미국(US)의 수도도 워싱턴 D.C.입니다.'

In [7]:
from datetime import datetime

def get_today():
    return datetime.now().strftime('%b %d')

prompt = PromptTemplate(
    template='오늘 날짜는 {today}입니다. 오늘 생일인 유명인 {n}명을 생년월일과 함께 나열해 주세요',
    input_variables=['n'],
    partial_variables={
        'today': get_today
    }
)

prompt.format(n=3)

'오늘 날짜는 Sep 02입니다. 오늘 생일인 유명인 3명을 생년월일과 함께 나열해 주세요'

In [8]:
llm = ChatOpenAI(model='gpt-5-nano')

chain = prompt | llm
# 오늘 날짜 기준
print(chain.invoke({'n': 3}).content)
print('===')
print(chain.invoke({'n': 3, 'today': 'Jan 2'}).content)

오늘은 9월 2일이네요. Sep 2 생일인 유명인 3명을 생년월일과 함께 바로 나열해 드리려면 최신 정보를 확인해야 하는데, 제 기억으로는 Salma Hayek가 확실히 1966년 9월 2일생입니다. 나머지 두 명은 정확하게 확인해 드리려면 인터넷 검색이 필요합니다. 원하시면 제가 신뢰 가능한 출처를 바탕으로 세 명 모두의 생년월일을 확인해 바로 알려드리겠습니다. 검색 허용해 주시겠어요?
===
오늘이 1월 2일이라는 가정으로, 1월 2일에 생일인 유명인 두 명과 생년월일을 먼저 적어 드립니다. 세 번째 인물은 정확성을 위해 인터넷 확인이 필요합니다.

- Isaac Asimov — 1920년 1월 2일
  미국의 과학소설가이자 과학 커뮤니케이터

- Jack Hanna — 1947년 1월 2일
  미국의 동물학자이자 TV 호스트

세 번째 인물도 확실히 알려드리려면 웹에서 확인해 보겠습니다. 원하시면 지금 바로 찾아서 세 번째를 포함한 완전한 목록으로 드릴게요.


## `ChatPromptTemplate`
- 채팅을 주고받는 템플릿 생성용
- 대화 목록을 LLM에게 주입
- 하나의 Chat 은 `role` 과 `message` 로 구성됨

In [9]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_template('{country}의 수도는 어디?')
chat_prompt.format(country='한국')

'Human: 한국의 수도는 어디?'

In [10]:
chat_template = ChatPromptTemplate.from_messages(
    [
        # role - message
        ('system', '당신은 친절한 AI어시스트. 이름은 {name} 야.'),
        ('human', '반가워!'),
        ('ai', '무엇을 도와드릴까요?'),
        ('human', '{user_input}')
    ]
)

# format vs format_messages
messages_str = chat_template.format(name='gaida', user_input='이름이 뭐니?')
messages_cls = chat_template.format_messages(name='gaida', user_input='이름이 뭐니?')
print(messages_str)
print(messages_cls)

System: 당신은 친절한 AI어시스트. 이름은 gaida 야.
Human: 반가워!
AI: 무엇을 도와드릴까요?
Human: 이름이 뭐니?
[SystemMessage(content='당신은 친절한 AI어시스트. 이름은 gaida 야.', additional_kwargs={}, response_metadata={}), HumanMessage(content='반가워!', additional_kwargs={}, response_metadata={}), AIMessage(content='무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={}), HumanMessage(content='이름이 뭐니?', additional_kwargs={}, response_metadata={})]


In [11]:
# 한덩어리 텍스트
llm.invoke(messages_str).content
# 실제 대화 내역으로 인지 (Langsmith 가서 확인)
llm.invoke(messages_cls).content

'제 이름은 gaida예요. 반가워요! 무엇을 도와드릴까요?'

In [12]:
chain = chat_template | llm | StrOutputParser()

chain.invoke({'name': '가이다', 'user_input': '내 이름은 유태영. 너와 나의 이름점을 봐줘'})

'반갑습니다, 유태영님! 가이다와의 이름 궁합을 재미로 살펴볼게요.\n\n- 점수: 82/100\n\n- 왜 이렇게 보이나요?\n  - 음성 조화: 유-태-영 vs 가-이-다의 시작 소리와 모음이 서로 다른 느낌이지만, 전체 흐름이 빠르고 명확해 서로를 보완하는 리듬을 만듭니다.\n  - 균형감: 모음과 자음이 번갈아 나와 안정감과 선명함이 함께 느껴집니다. 발음하기도 비교적 쉽고 기억에 남는 조합이에요.\n  - 분위기/이미지: 가이다가 안내자 같은 이미지를 주면, 유태영님의 다채로운 음색(유연함 + 강인함)과 어울려 긍정적이고 활기찬 이미지를 만들어 냅니다.\n\n- 간단한 활용 팁\n  - 이름의 이미지를 더 잘 맞추고 싶다면, 글자별 시작과 끝 소리에 맞춘 짧은 문장 연습을 해보면 좋습니다. 예: “유태영은 가이드를 따라 새로운 길을 밝힌다” 같은 식으로 서로의 이미지를 연결해 보는 식으로요.\n\n다른 방식으로 더 깊게 보고 싶으신가요? 예를 들어 획수 기반의 이름 점이나 성격 궁합 스타일로도 살펴드릴 수 있어요. 또는 다른 이름과의 궁합도 해볼까요?'

## 프롬프팅 방법
내가 원하는 답변을 구구절절 설명 X -> 예시를 주고 답변을 형태를 유도
- Zero-shot prompting : 예시 없이 질문만 -> 바로 답변
- One-shot prompting  : 예시 1개 + 질문 -> 모델이 예시를 모방해서 답변
- **Few-shot prompting**  : 예시 여러개 + 질문 -> 예시들의 패턴을 일반화 해서 답변

In [13]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.output_parsers import StrOutputParser

reasoning = {
    "effort": "low",  # 'low', 'medium', or 'high'
    "summary": "auto",  # 'detailed', 'auto', or None
}

llm = ChatOpenAI(model='gpt-4.1-nano')

examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 스티브 잡스는 몇 살에 사망했나요?
중간 답변: 스티브 잡스는 56세에 사망했습니다.
추가 질문: 아인슈타인은 몇 살에 사망했나요?
중간 답변: 아인슈타인은 76세에 사망했습니다.
최종 답변은: 아인슈타인
""",
    },
    {
        "question": "네이버의 창립자는 언제 태어났나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 네이버의 창립자는 누구인가요?
중간 답변: 네이버는 이해진에 의해 창립되었습니다.
추가 질문: 이해진은 언제 태어났나요?
중간 답변: 이해진은 1967년 6월 22일에 태어났습니다.
최종 답변은: 1967년 6월 22일
""",
    },
    {
        "question": "율곡 이이의 어머니가 태어난 해의 통치하던 왕은 누구인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 율곡 이이의 어머니는 누구인가요?
중간 답변: 율곡 이이의 어머니는 신사임당입니다.
추가 질문: 신사임당은 언제 태어났나요?
중간 답변: 신사임당은 1504년에 태어났습니다.
추가 질문: 1504년에 조선을 통치한 왕은 누구인가요?
중간 답변: 1504년에 조선을 통치한 왕은 연산군입니다.
최종 답변은: 연산군
""",
    },
    {
        "question": "올드보이와 기생충의 감독이 같은 나라 출신인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 올드보이의 감독은 누구인가요?
중간 답변: 올드보이의 감독은 박찬욱입니다.
추가 질문: 박찬욱은 어느 나라 출신인가요?
중간 답변: 박찬욱은 대한민국 출신입니다.
추가 질문: 기생충의 감독은 누구인가요?
중간 답변: 기생충의 감독은 봉준호입니다.
추가 질문: 봉준호는 어느 나라 출신인가요?
중간 답변: 봉준호는 대한민국 출신입니다.
최종 답변은: 예
""",
    },
]

example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"

chain = prompt | llm | StrOutputParser()

res = chain.invoke({'question': question})

print(res)

이 질문에 추가 질문이 필요한가요: 예.  
추가 질문: Google이 창립된 연도는 언제인가요?  
중간 답변: Google은 1998년에 창립되었습니다.  
추가 질문: Bill Gates는 1998년 얼마 살았나요?  
중간 답변: Bill Gates는 1998년 1월 31일생이므로, 1998년에는 1998 - 1955 = 43세였습니다.  
최종 답변은: 43살입니다.


## `langchain-hub`
다양한 사용자들이 업로드한 프롬프트를 받아서 활용

In [14]:
from langchain import hub

prompt = hub.pull('hwchase17/react')
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
